# പാഠം 13 - ഏജന്‍റ് മെമ്മറി


## സജ്ജീകരണം

ഈ നോട്ട്ബുക്ക് **Microsoft Agent Framework** (MAF) ഉപയോഗിച്ച് **സ്ഥിരതയുള്ള മെമ്മറിയുമായി** ഒരു യാത്രാ ബുക്കിംഗ് ഏജന്റ് നിർമ്മിക്കുന്ന വിധം കാണിക്കുന്നു.

ഏജെന്റിന്റെ വിവിധ മെമ്മറി തരം — വർക്ക്, ഷോർട്ട്-ടേം, ലോങ്ങ്-ടേം — ഏജന്റ് ഒരു സംഭാഷണത്തിൽ தகவൽ എങ്ങനെ നിലനിർത്തുകയും ഉപയോഗിക്കുകയും ചെയ്യുമെന്ന് നിങ്ങൾക്ക് പഠിക്കാൻ സാധിക്കും.

**ആവശ്യമായവ:**
- ഡിപ്ലോയ് ചെയ്ത ചാറ്റ് മോഡൽ (ഉദാ. `gpt-4o-mini`) ഉള്ള ഒരു Azure AI Foundry പ്രോജക്ട്.
- Azure CLIലേക്ക് ലോഗിൻ ചെയ്ത് — നിങ്ങളുടെ ടെർമിനലിൽ `az login` പ്രവർത്തിപ്പിക്കുക.
- `AZURE_AI_PROJECT_ENDPOINT` — നിങ്ങളുടെ Azure AI Foundry പ്രോജക്ട് എന്റർപോയിന്റ്.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — നിങ്ങളുടെ ഡിപ്ലോയ് ചെയ്ത മോഡലിന്റെ പേര്.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ഏജന്റ് മെമ്മറി പ്രകാര്യങ്ങൾ

AI ഏജന്റുകൾ വ്യത്യസ്ത തരം മെമ്മറികൾ ഉപയോഗിക്കാം, ഓരോതും വ്യത്യസ്തമായ ഒരു ഉദ്ദേശ്യം സേവിക്കുന്നു:

### വർകിംഗ് മെമ്മറി  
സന്ദേശങ്ങൾ ഒരു സെഷനിൽ കൈമാറുന്ന സംഭാഷണ ത്രെഡ് തന്നെ. ഏജന്റ് സാദൃശ്യം നിലനിർത്താൻ ഒരേ ത്രെഡിലെ മുൻവാഗ്ദാനങ്ങൾക്കു പിന്നിൽ തിരിച്ചു നോക്കാം. MAF-ൽ ഇത് **`agent.create_session()`** വഴി സൃഷ്ടിച്ചുതരം ആണ്, ഇത് `AgentSession` ആണ് തിരിച്ചുവരുന്നത്.

### ഷോർട്ട്-ടേം മെമ്മറി  
ഒരു ടാസ്കിനോ സെഷനോ നടക്കുന്നതിനിടെ നിലനിൽക്കുന്ന വിവരങ്ങൾ, എന്നാൽ സ്ഥിരമായി സൂക്ഷിക്കുന്നതല്ല. ഉദാഹരണത്തിന്, ഏജന്റ് ഒരു ബഹു-ടേൺ പ്ലാനിംഗ് സംഭാഷണത്തിനിടെ കാര്യങ്ങൾ സംഭരിച്ച് അവ ഉപയോഗിച്ച് ഒരു അന്തിമ യാത്രാസൂചിക നിർമ്മിക്കാം.

### ലോംഗ്-ടേം മെമ്മറി  
**സെഷനുകൾക്കിടയിലെ** മുൻഗാമി പെരുമാറ്റങ്ങളും വിവരങ്ങളും. പുനരാഗമിക്കുന്ന ഉപയോക്താവ് തന്റെ ഭക്ഷണ പരിധികളും യാത്ര ശൈലിയും ആവർത്തിച്ച് പറയേണ്ടതില്ല. ലാംഗ്-ടേം മെമ്മറിയെ സാധാരണയായി ഒരു ബാഹ്യ സംഭരണ സംവിധാനമാണ് പിന്തുണയ്ക്കുന്നത് — ഒരു ഡേറ്റാബേസോ, ഫയൽ അഥവാ വെക്ടർ ഇൻഡക്സോ — ഇത് ടൂളുകൾ വഴി ഏജന്റിന് ലഭ്യമാക്കുന്നു.


## സെഷനുകളോട് കൂടിയ പ്രവർത്തന സ്മृति

സ്മൃതിയുടെ ഏറ്റവും ലളിതമായ രൂപമാണ് സംഭാഷണ സെഷൻ. ഒരു സമാന സെഷൻ (`agent.create_session()` വഴി നിർമ്മിച്ച) നിരന്തരമായ `agent.run()` കോൾസ്‌ക്ക് നൽകുമ്പോൾ, ഏജന്റ് ആ സംഭാഷണത്തിന്റെ പൂർണ്ണ ചരിത്രം കാണുകയും മുൻ സ്മരണകൾ ഓർക്കുകയും ചെയ്യുന്നു.

ഒരു യാത്രാ ഏജന്റ് സൃഷ്‌ടിച്ച് പ്രവർത്തന സ്മൃതി പ്രകടിപ്പിക്കാം.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ഏജന്റ് ബജറ്റ് ശരിയായി ഓർമ്മിച്ചു കൂടിയുള്ള സന്ദേശങ്ങൾക്കും ഒരേ സെഷനുണ്ടായതിനാൽ. ഇതാണ് **പ്രവർത്തന ഓർമ്മ** — ഇത് സെഷന്റെ ആയുസ്സിനുള്ളതായാണ് നിലനിൽക്കുന്നത്.

### പുതിയ ത്രെഡുമായി എന്തു സംഭവിക്കുന്നു?

ഞങ്ങൾ ഒരു **പുതിയ** സെഷൻ സൃഷ്ടിച്ചാൽ, ഏജന്റിന് മുൻവന്താലിസംഭാഷണത്തിന്റെയോ ഓർമ്മ ഉണ്ടാകില്ല:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## ദീർഘകാല സ്മൃതി മാതൃക

ഉപയോക്തൃ ഇഷ്ടങ്ങളെ **സെഷനുകളുടെ വഴി** ഓർത്തുപിടിക്കാനായി, സംഭാഷണ ത്രെഡിന്റെ പുറത്തുള്ള സ്ഥിരമായ സംഭരണം ആവശ്യമുണ്ട്. ഏജന്റ് ഈ സംഭരണം **ഉപകരണങ്ങൾ** വഴി ആക്‌സസ് ചെയ്യുന്നു — വിവരങ്ങൾ സൂക്ഷിക്കാനും തിരിച്ചെടുക്കാനുമുള്ള ഫംഗ്ഷനുകൾ.

താഴെ ഒരു ലളിതമായ ഇൻ-മെമ്മറി പ്രിഫറൻസ് സ്റ്റോർ (പ്രോഡക്ഷനിൽ ഇത് ഡാറ്റാബേസ് അല്ലെങ്കിൽ വെക്ടർ ഇൻഡക്സ് കൊണ്ടു പിന്തുടരാം) നടപ്പിലാക്കി, ഏജന്റ് ഉപയോഗിക്കാവുന്ന ഉപകരണങ്ങളായി വരുന്ന തരത്തിൽ അതിനെ എക്സ്പോസ് ചെയ്യുന്നു.

### ആർക്കിടെക്ടർ
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### സാഹചര്യo 1 — ആദ്യം വാര്‍ത്താവകാശി വാര്‍ത്താവകാശി ഒരു വാർഷിക യാത്ര ബുക്ക് ചെയ്യുന്നു

സാറാ ആദ്യമായി സന്ദര്‍ശിക്കുന്നു. ഏജന്റ് ത	required.preferences ഉപകരണങ്ങളിലൂടെ സൂക്ഷിച്ച് അവ ഹോട്ടലുകൾ ശുപാർശ ചെയ്യാൻ ഉപയോഗിക്കണം.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### അഭിജ്ഞാപകം 2 — സാറായ്ക്ക് ചില ആഴ്ചകൾക്കു ശേഷം മടങ്ങിയാണ് വരുന്നത്

സാറായി ഒരു **പുതിയ ത്രെഡ്** തുടങ്ങുന്നു (പുതിയ സെഷൻ അനുകരിച്ചുകൊണ്ട്). പ്രഥമ സ്മൃതി ശൂന്യമാണ്, പക്ഷേ ദീർഘകാല ആഗ്രഹ സംഭരണം ഇപ്പോഴും അവളുടെ വിവരങ്ങൾ സൂക്ഷിച്ചിട്ടുണ്ട്. ഏജന്റിന് അത് തിരികെ കണ്ടെത്തി ശുപാർശകൾ വ്യക്തിഗതമായി ഒരുക്കാൻ അത് ഉപയോഗിക്കണം.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## സാമരിച്ചു

ഈ പാഠത്തിൽ നിങ്ങൾ മൂന്ന് തരത്തിലുള്ള ഏജന്റ് മെമ്മറിയും മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്കിൽ അവ എങ്ങനെ നടപ്പിലാക്കാമെന്നതും പഠിച്ചു:

| മെമ്മറി തരം | MAF സംവിധാനം | കാലാവധി |
|---|---|---|
| **വർക്കിങ്** | `agent.create_session()` | ഒരു സംഭാഷണം മാത്രം |
| **ഷോർട്ട്-ടേം** | ഒരു ത്രെഡിനുള്ളിൽ സമാഹരിച്ച ചുരുക്കം | ഒരു ടാസ്‌ക് / സെഷൻ മാത്രം |
| **ലോങ്ങ്-ടേം** | `@tool` ഫംഗ്ഷനുകളിലൂടെ പ്രവേശനമുള്ള ബാഹ്യ സംഭരണം | സെഷനുകൾക്കു മുകളില്‍ |

### പ്രധാനപ്പെട്ട കാര്യങ്ങൾ
1. **`agent.create_session()`** പ്രവർത്തന മെമ്മറി നൽകുന്നു — ഏജന്റ് ഒരു സെഷനിൽ മുഴുവൻ സംഭാഷണ ചരിത്രവും കാണുന്നു.
2. **പുതിയ സെഷനുകൾക്ക് ചുരുക്കം ഇല്ല** — ലോങ്ങ്-ടേം മെമ്മറി ഇല്ലെങ്കിൽ ഏജന്റ് മുമ്പത്തെ സംഭാഷണങ്ങൾ ഓർക്കാൻ കഴിയില്ല.
3. **`@tool` ഫംഗ്ഷനുകൾ** ഇട വിടുന്നു — ഇവ ഏജന്റ് സ്ഥിരതയുള്ള സംഭരണിയിൽ നിന്ന് വിവരങ്ങൾ സംരക്ഷിക്കുകയും പുനഃപ്രാപിക്കയും ചെയ്യാൻ അനുവദിക്കുന്നു.
4. **വ്യക്തിഗതമാക്കൽ സമയം കൂടിയപ്പോൾ മെച്ചപ്പെടുന്നു** — ഓരേറെ കാര്യം സൂക്ഷിച്ചതോടെ ഏജന്റിന്റെ ശുപാർശകൾ മെച്ചപ്പെടുന്നു.

### യാഥാർഥ്യ wereldാനുഭവങ്ങൾ
- **കസ്റ്റമർ സർവീസ്**: ഉപഭോക്താവിന്റെ ചരിത്രവും മുൻഗണനകളും ഓർക്കുക
- **പേഴ്സണൽ അസിസ്റ്റന്റുകൾ**: ദിവസങ്ങളിലോ ആഴ്ചകളിലോ ഉള്ള ചുരുക്ക് സൂക്ഷിക്കുക
- **ആരോഗ്യപരിചരണം**: രോഗിയുടെ വിവരങ്ങളും മുൻഗണനകളും ട്രാക്ക് ചെയ്യുക
- **ഇ-കോമേഴ്സ്**: ചരിത്രത്തെ ആശ്രയിച്ച് വ്യക്തിഗത ഷോപ്പിംഗ്

### അടുത്ത ഘട്ടങ്ങൾ
- ഇൻ-മെമ്മറി ഡിക്ഷണറിയെയും ഡാറ്റാബേസ് അല്ലെങ്കിൽ വെക്ടർ സ്റ്റോറിൽ (ഉദാ: Azure AI Search) മാറ്റുക
- സമയം പ്രസക്തമായ വിവരങ്ങൾക്ക് മെമ്മറി കാലഹരണപ്പെടൽ ചേർക്കുക
- പങ്കിടുന്ന മെമ്മറിയുള്ള മൾട്ടി-ഏജന്റ് സിസ്റ്റങ്ങൾ നിർമ്മിക്കുക
- അറിവു-ഗ്രാഫ് പിന്തുണയുള്ള Cognee നോൺബുക്ക് അന്വേഷിക്കുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**പറഞ്ഞുകൊള്ളൽ**:
ഈ രേഖ AI പരിഭാഷ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. നാം ശരിയായ വിവർത്തനത്തിന് ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ തെറ്റുകൾ അല്ലെങ്കിൽ അശുദ്ധികൾ ഉണ്ടാകാമെന്ന് ദയവായി തിരിച്ചറിയുക. അതിൻറെ മാതൃഭാഷയിലെ യഥാർത്ഥ രേഖ അധികാരമുള്ള ഉറവിടമായി പരിഗണിക്കേണ്ടതാണ്. നിർണ്ണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മാനവ പരിഭാഷ ശുപാർശ ചെയ്യപ്പെടുന്നു. ഈ വിവർത്തനത്തിന്റെ ഉപയോഗത്തിൽ നിന്നുണ്ടാകുന്ന ഏതെങ്കിലും തെറ്റിദ്ധാരണകൾക്കും തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കും ഞങ്ങൾ ബാധ്യത വഹിക്കുന്നില്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
